In [1]:
# time：计时工具，可统计模型训练耗时
import time

# sklearn.datasets：内置/下载数据集
# load_iris：鸢尾花分类数据集；fetch_20newsgroups：新闻文本数据集；fetch_california_housing：加州房价回归数据集
from sklearn.datasets import load_iris, fetch_20newsgroups, fetch_california_housing

# model_selection：模型选择、数据集划分、网格搜索调参
# train_test_split：划分训练集、测试集
# GridSearchCV：网格搜索交叉验证，寻找最优超参数
from sklearn.model_selection import train_test_split, GridSearchCV

# K近邻分类器 KNN
from sklearn.neighbors import KNeighborsClassifier

# StandardScaler：Z‑score标准化，均值0方差1
from sklearn.preprocessing import StandardScaler

# TfidfVectorizer：文本TF‑IDF特征提取
from sklearn.feature_extraction.text import TfidfVectorizer

# 多项式朴素贝叶斯，多用于文本分类（multinomial naive bayes）
from sklearn.naive_bayes import MultinomialNB

# classification_report：分类评估报告，输出精确率、召回率、F1
from sklearn.metrics import classification_report

# DictVectorizer：字典数据转特征矩阵
from sklearn.feature_extraction import DictVectorizer

# DecisionTreeClassifier决策树分类器；export_graphviz导出决策树dot文件，用于可视化（graphviz = graph visualization）
from sklearn.tree import DecisionTreeClassifier, export_graphviz

# RandomForestClassifier随机森林分类器，集成学习，多棵决策树
from sklearn.ensemble import RandomForestClassifier

# pandas：表格数据处理
import pandas as pd
# numpy：数值计算、数组操作
import numpy as np

# roc_auc_score：AUC指标，二分类模型评估
# ROC：Receiver Operating Characteristic，受试着工作特征曲线；AUC：Area Under Curve，曲线下面积
from sklearn.metrics import roc_auc_score

## load直接加载到内存的，数据集比较小，并不会保存到本地磁盘fetch数据集比较大，下载下来后会存在本地磁盘，下一次就不会再连接sklearn的服务器

In [2]:
# 鸢尾花数据集，查看特征，目标，样本量
iris = load_iris()

print('获取特征值：')
print(type(iris.data))                    # <class 'numpy.ndarray'>
print('--' * 50)
print(iris.data.shape)                    # (150, 4)
iris.data

获取特征值：
<class 'numpy.ndarray'>
----------------------------------------------------------------------------------------------------
(150, 4)


array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
       [4.9, 3

In [3]:
print(type(iris))

<class 'sklearn.utils._bunch.Bunch'>


In [7]:
from importlib import resources
import sklearn.datasets.data as data_module
"""
    读取sklearn包内部自带的原始iris.scv磁盘文件路径
    resources.files(data_module)：获取sklearn.datasets.data这个包在磁盘上的目录对象
    / "iris.csv"：路径拼接，指向包内部存放的原始iris.csv文件
    data_path得到的是磁盘真实文件路径
"""
data_path = resources.files(data_module) / "iris.csv"
print(data_path)
# /Users/qingjiabu/Library/Python/3.12/lib/python/site-packages/sklearn/datasets/data/iris.csv
print('--' * 50)

with data_path.open("r") as f:
    lines = f.readlines()

# 第一行是元信息，load_iris会跳过第一行
meta_line = lines[0]
sample_lines = lines[1:2]
print("sklearn包内iris.csv第一行元数据：", meta_line)
print(sample_lines)
print(len(sample_lines))
print(type(sample_lines))

/Users/qingjiabu/Library/Python/3.12/lib/python/site-packages/sklearn/datasets/data/iris.csv
----------------------------------------------------------------------------------------------------
sklearn包内iris.csv第一行元数据： 150,4,setosa,versicolor,virginica

['5.1,3.5,1.4,0.2,0\n']
1
<class 'list'>


In [14]:
# as_file拿到真实磁盘Path对象
with resources.as_file(resources.files(data_module)/"iris.csv") as real_path:
    print("iris.csv真实磁盘路径：", real_path)
    # 直接读取原始csv全部文本
    text = real_path.read_text(encoding="utf-8")

print("========磁盘原始iris.csv全部内容========")
"""
setosa,versicolor,virginica：
    鸢尾花数据集3个分类标签（3种花品种）
    setosa：山鸢尾 -> 标签数字0
    versicolor：变色鸢尾 -> 标签数字1
    virginica：维吉尼亚鸢尾 -> 标签数字2
"""
print(text)

iris.csv真实磁盘路径： /Users/qingjiabu/Library/Python/3.12/lib/python/site-packages/sklearn/datasets/data/iris.csv
========磁盘原始iris.csv全部内容========
150,4,setosa,versicolor,virginica
5.1,3.5,1.4,0.2,0
4.9,3.0,1.4,0.2,0
4.7,3.2,1.3,0.2,0
4.6,3.1,1.5,0.2,0
5.0,3.6,1.4,0.2,0
5.4,3.9,1.7,0.4,0
4.6,3.4,1.4,0.3,0
5.0,3.4,1.5,0.2,0
4.4,2.9,1.4,0.2,0
4.9,3.1,1.5,0.1,0
5.4,3.7,1.5,0.2,0
4.8,3.4,1.6,0.2,0
4.8,3.0,1.4,0.1,0
4.3,3.0,1.1,0.1,0
5.8,4.0,1.2,0.2,0
5.7,4.4,1.5,0.4,0
5.4,3.9,1.3,0.4,0
5.1,3.5,1.4,0.3,0
5.7,3.8,1.7,0.3,0
5.1,3.8,1.5,0.3,0
5.4,3.4,1.7,0.2,0
5.1,3.7,1.5,0.4,0
4.6,3.6,1.0,0.2,0
5.1,3.3,1.7,0.5,0
4.8,3.4,1.9,0.2,0
5.0,3.0,1.6,0.2,0
5.0,3.4,1.6,0.4,0
5.2,3.5,1.5,0.2,0
5.2,3.4,1.4,0.2,0
4.7,3.2,1.6,0.2,0
4.8,3.1,1.6,0.2,0
5.4,3.4,1.5,0.4,0
5.2,4.1,1.5,0.1,0
5.5,4.2,1.4,0.2,0
4.9,3.1,1.5,0.2,0
5.0,3.2,1.2,0.2,0
5.5,3.5,1.3,0.2,0
4.9,3.6,1.4,0.1,0
4.4,3.0,1.3,0.2,0
5.1,3.4,1.5,0.2,0
5.0,3.5,1.3,0.3,0
4.5,2.3,1.3,0.3,0
4.4,3.2,1.3,0.2,0
5.0,3.5,1.6,0.6,0
5.1,3.8,1.9,0.4,0
4.8,3.0,1.4,0.

In [17]:
print('目标值：')
print(iris.target)
print('--' * 50)

print(iris.DESCR)
print('--' * 50)

# sepal 萼片；花萼   petal 花瓣
# ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
print(iris.feature_names)

print('--' * 50)
print(iris.target_names)                    # ['setosa' 'versicolor' 'virginica']

目标值：
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2]
----------------------------------------------------------------------------------------------------
.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Statistics:

============== ==== ==== ======= ===== ====================
                Min  Max   Mean    SD   Class Correlation
========

In [9]:
# test_size=0.25：测试集占总样本25%（共38条数据），训练集占75%（共112条数据）；random_state=1：随机种子。固定这个数字，每次运行划分出来的训练、测试集完全一摸一样，可复现实验
# x_train：训练集特征（feature_names）；x_test：测试集特征；y_train：训练集标签（target_names）；y_test：测试集标签
# 注意返回值顺序
x_train, x_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.25, random_state=1)

# x_train: feature_values   y_train: target_values
print('训练集特征值和目标值：\n', x_train, y_train)
print('训练集特征值shape：', x_train.shape)                            # (112, 4)
print('--' * 50)

# x_test: feature_values   y_test: target_values
print('测试集特征值和目标值：\n', x_test, y_test)
print('测试集特征值shape：', x_test.shape)                             # (38, 4)

训练集特征值和目标值：
 [[6.5 2.8 4.6 1.5]
 [6.7 2.5 5.8 1.8]
 [6.8 3.  5.5 2.1]
 [5.1 3.5 1.4 0.3]
 [6.  2.2 5.  1.5]
 [6.3 2.9 5.6 1.8]
 [6.6 2.9 4.6 1.3]
 [7.7 2.6 6.9 2.3]
 [5.7 3.8 1.7 0.3]
 [5.  3.6 1.4 0.2]
 [4.8 3.  1.4 0.3]
 [5.2 2.7 3.9 1.4]
 [5.1 3.4 1.5 0.2]
 [5.5 3.5 1.3 0.2]
 [7.7 3.8 6.7 2.2]
 [6.9 3.1 5.4 2.1]
 [7.3 2.9 6.3 1.8]
 [6.4 2.8 5.6 2.2]
 [6.2 2.8 4.8 1.8]
 [6.  3.4 4.5 1.6]
 [7.7 2.8 6.7 2. ]
 [5.7 3.  4.2 1.2]
 [4.8 3.4 1.6 0.2]
 [5.7 2.5 5.  2. ]
 [6.3 2.7 4.9 1.8]
 [4.8 3.  1.4 0.1]
 [4.7 3.2 1.3 0.2]
 [6.5 3.  5.8 2.2]
 [4.6 3.4 1.4 0.3]
 [6.1 3.  4.9 1.8]
 [6.5 3.2 5.1 2. ]
 [6.7 3.1 4.4 1.4]
 [5.7 2.8 4.5 1.3]
 [6.7 3.3 5.7 2.5]
 [6.  3.  4.8 1.8]
 [5.1 3.8 1.6 0.2]
 [6.  2.2 4.  1. ]
 [6.4 2.9 4.3 1.3]
 [6.5 3.  5.5 1.8]
 [5.  2.3 3.3 1. ]
 [6.3 3.3 6.  2.5]
 [5.5 2.5 4.  1.3]
 [5.4 3.7 1.5 0.2]
 [4.9 3.1 1.5 0.2]
 [5.2 4.1 1.5 0.1]
 [6.7 3.3 5.7 2.1]
 [4.4 3.  1.3 0.2]
 [6.  2.7 5.1 1.6]
 [6.4 2.7 5.3 1.9]
 [5.9 3.  5.1 1.8]
 [5.2 3.5 1.5 0.2]
 [5.1 3.3 1.7 0.5]

In [19]:
150 * 0.25

37.5

In [35]:
"""
    fetch_20newsgroups：下载20个新闻组文本数据集，属于远程下载型数据集（fetch_开头），不是内置在sklearn包里，需要从网络下载，保存到本地缓存文件夹
    subset='all'：全部数据集，train+test合并在一起；subset='train'：只取训练部分；subset='test'：只取测试部分
    data_home='data'：指定数据集下载后缓存存放的文件夹名，会在你的项目下创建./data/20news_home/，把新闻文本压缩包解压存这里
    如果不写data_home，默认放在用户目录下~/scikit_learn_data
"""
news = fetch_20newsgroups(subset='all', data_home='data')

# print(news.feature_names)           这个数据集没有特征名，因为没有特征，只有文本数据（样本数据是一个文本，没有特征）
# print(news.DESCR)
print(news.data[0])
print('--' * 50)
print(type(news.data))
# print(news.data.shape)            .shape是numpy ndarray的属性，python内置list中没有该属性
print('--' * 50)
print(news.target[0: 15])
print('--' * 50)
from pprint import pprint
pprint(list(news.target_names))

From: Mamatha Devineni Ratnam <mr47+@andrew.cmu.edu>
Subject: Pens fans reactions
Organization: Post Office, Carnegie Mellon, Pittsburgh, PA
Lines: 12
NNTP-Posting-Host: po4.andrew.cmu.edu



I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a lot of
fun in the next couple of games since the Pens are going to beat the pulp out of Jersey anyway. I was very disappointed not to see the Islanders lose the final
regular season game.          PENS RULE!!!


-------------------------------------------------------------------------------------------------

In [27]:
print(news.data[2])

From: hilmi-er@dsv.su.se (Hilmi Eren)
Subject: Re: ARMENIA SAYS IT COULD SHOOT DOWN TURKISH PLANES (Henrik)
Lines: 95
Nntp-Posting-Host: viktoria.dsv.su.se
Reply-To: hilmi-er@dsv.su.se (Hilmi Eren)
Organization: Dept. of Computer and Systems Sciences, Stockholm University




|>The student of "regional killings" alias Davidian (not the Davidian religios sect) writes:


|>Greater Armenia would stretch from Karabakh, to the Black Sea, to the
|>Mediterranean, so if you use the term "Greater Armenia" use it with care.


	Finally you said what you dream about. Mediterranean???? That was new....
	The area will be "greater" after some years, like your "holocaust" numbers......




|>It has always been up to the Azeris to end their announced winning of Karabakh 
|>by removing the Armenians! When the president of Azerbaijan, Elchibey, came to 
|>power last year, he announced he would be be "swimming in Lake Sevan [in 
|>Armeniaxn] by July".
		*****
	Is't July in USA now????? Here in Sweden it's

In [34]:
print('总共新闻数据样本（条）：', len(news.data))
print('--' * 50)
print('所有新闻标签（即新闻的种类数）：', news.target)
print('--' * 50)
print(min(news.target), max(news.target))                       # 最小标签为0，最大标签为19，表明新闻分为20类
print(len(news.target_names))                                   # 也可以直接返回标签名长度表明新闻总类别数

总共新闻数据样本（条）： 18846
----------------------------------------------------------------------------------------------------
所有新闻标签（即新闻的种类数）： [10  3 17 ...  3  1  7]
----------------------------------------------------------------------------------------------------
0 19
20


In [41]:
house = fetch_california_housing(data_home='data')
print('获取特征值：', house.data[0])
print('样本的形状：', house.data.shape)
print(type(house.data))                                         # <class 'numpy.ndarray'>
print('--' * 50)

print('目标值：', house.target)

# ['MedHouseVal']
print(house.target_names)
print('--' * 50)

print(house.DESCR)
print('--' * 50)

# ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
# 共有8个特征
print(house.feature_names)                  # Number of Attributes

获取特征值： [   8.3252       41.            6.98412698    1.02380952  322.
    2.55555556   37.88       -122.23      ]
样本的形状： (20640, 8)
<class 'numpy.ndarray'>
----------------------------------------------------------------------------------------------------
目标值： [4.526 3.585 3.521 ... 0.923 0.847 0.894]
['MedHouseVal']
----------------------------------------------------------------------------------------------------
.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude  

# 2. 分类估计器

In [42]:
np.sqrt(15 * 15 + 14 * 14)

np.float64(20.518284528683193)

In [15]:
"""
K-近邻预测用户签到位置
"""
data = pd.read_csv('./data/FBlocation/train.csv')

print(data.head())
print('--' * 50)
print(data.shape)
print('--' * 50)
print(data.info())
print('--' * 50)
# 处理数据：1.缩小数据，查询数据，减少计算时间
data = data.query('x > 1.0 & x < 1.25 & y > 2.5 & y < 2.75')
data

   row_id       x       y  accuracy    time    place_id
0       0  0.7941  9.0809        54  470702  8523065625
1       1  5.9567  4.7968        13  186555  1757726713
2       2  8.3078  7.0407        74  322648  1137537235
3       3  7.3665  2.5165        65  704587  6567393236
4       4  4.0961  1.1307        31  472130  7440663949
----------------------------------------------------------------------------------------------------
(29118021, 6)
----------------------------------------------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 29118021 entries, 0 to 29118020
Data columns (total 6 columns):
 #   Column    Dtype  
---  ------    -----  
 0   row_id    int64  
 1   x         float64
 2   y         float64
 3   accuracy  int64  
 4   time      int64  
 5   place_id  int64  
dtypes: float64(2), int64(4)
memory usage: 1.3 GB
None
----------------------------------------------------------------------------------------------------


,row_id,x,y,accuracy,time,place_id
600,600,1.2214,2.7023,17,65380,6683426742
957,957,1.1832,2.6891,58,785470,6683426742
4345,4345,1.1935,2.6550,11,400082,6889790653
4735,4735,1.1452,2.6074,49,514983,6822359752
5580,5580,1.0089,2.7287,19,732410,1527921905
...,...,...,...,...,...,...
29100203,29100203,1.0129,2.6775,12,38036,3312463746
29108443,29108443,1.1474,2.6840,36,602524,3533177779
29109993,29109993,1.0240,2.7238,62,658994,6424972551
29111539,29111539,1.2032,2.6796,87,262421,3533177779


In [16]:
# data['time']是上面data数据time列
"""
作用：把Unix时间戳（秒数）转为pandas的datetime时间类型
    data['time']: 一列数字，代表从1970-01-01 00:00:00 UTC 开始累计的秒数(时间戳)
    unit='s' -> unit=second，输入单位是秒（数字代表距离起点一共经过多少秒）
    返回：datatime64 类型的Series，可以做时间运算、切片、排序
"""
time_value = pd.to_datetime(data['time'], unit='s')
print(type(time_value))                                     # <class 'pandas.Series'>
print('--' * 50)
print(time_value.head(10))

<class 'pandas.Series'>
----------------------------------------------------------------------------------------------------
600    1970-01-01 18:09:40
957    1970-01-10 02:11:10
4345   1970-01-05 15:08:02
4735   1970-01-06 23:03:03
5580   1970-01-09 11:26:50
6090   1970-01-02 16:25:07
6234   1970-01-04 15:52:57
6350   1970-01-01 10:13:36
7468   1970-01-09 15:26:06
8478   1970-01-08 23:52:02
Name: time, dtype: datetime64[s]


In [17]:
# 此处转换后的time_value与上面的time_value顺序保持一致；后面可以补一句 data.index = time_value 使data数据按照日期索引
time_value = pd.DatetimeIndex(time_value)
"""
    这里的time_value索引改为了日期时间索引，但不代表日期时间索引是有序的，按照原始索引顺序排列，次序不变
"""
# 原始数据data['time']这一列本身就是乱序的，日期时间索引不改变样本顺序
print(time_value[0: 10])

DatetimeIndex(['1970-01-01 18:09:40', '1970-01-10 02:11:10',
               '1970-01-05 15:08:02', '1970-01-06 23:03:03',
               '1970-01-09 11:26:50', '1970-01-02 16:25:07',
               '1970-01-04 15:52:57', '1970-01-01 10:13:36',
               '1970-01-09 15:26:06', '1970-01-08 23:52:02'],
              dtype='datetime64[s]', name='time', freq=None)


In [18]:
data.shape

(17710, 6)

In [19]:
"""
DataFrame与ndarray的区别：
    ndarray：numpy数组，纯数值矩阵，numpy的数据结构
    DataFrame：pandas表格，带标签的二维表，pandas的数据结构
"""
print(type(data))

data.insert(data.shape[1], 'day', time_value.day)
data.insert(data.shape[1], 'hour', time_value.hour)
data.insert(data.shape[1], 'weekday', time_value.weekday)

data = data.drop(['time'], axis=1)
print('--' * 50)
data.head()

<class 'pandas.DataFrame'>
----------------------------------------------------------------------------------------------------


,row_id,x,y,accuracy,place_id,day,hour,weekday
600,600,1.2214,2.7023,17,6683426742,1,18,3
957,957,1.1832,2.6891,58,6683426742,10,2,5
4345,4345,1.1935,2.6550,11,6889790653,5,15,0
4735,4735,1.1452,2.6074,49,6822359752,6,23,1
5580,5580,1.0089,2.7287,19,1527921905,9,11,4


In [20]:
"""
pd.Period：时间段对象，代表一个时间区间，这里频率 'h'=hour小时，代表[1970-01-01 18点这一整个小时区间]
    区分：
        Timestamp：时间点（某个精确时刻）
        Period：时间片/时间段（例如1小时、1天、1个月）
    per.weekday: 返回星期几，数字：
        0 = 星期一
        1 = 星期二
        2 = 星期三
        3 = 星期四
        4 = 星期五
        5 = 星期六
        6 = 星期日
    1970-01-01是星期四，所以per.weekday = 3
"""
per = pd.Period('1970-01-01 18:00', 'h')
per.weekday

3

In [21]:
# 观察数据，查看是否有空值，异常值
data.describe()

,row_id,x,y,accuracy,place_id,day,hour,weekday
count,1.771000e+04,17710.000000,17710.000000,17710.000000,1.771000e+04,17710.000000,17710.000000,17710.000000
mean,1.450569e+07,1.122538,2.632309,82.482101,5.129895e+09,5.101863,11.485545,3.092377
std,8.353805e+06,0.077086,0.070144,113.613227,2.357399e+09,2.709287,6.932195,1.680218
min,6.000000e+02,1.000100,2.500100,1.000000,1.012024e+09,1.000000,0.000000,0.000000
25%,7.327816e+06,1.049200,2.573800,25.000000,3.312464e+09,3.000000,6.000000,2.000000
50%,1.443071e+07,1.123300,2.642300,62.000000,5.261906e+09,5.000000,12.000000,3.000000
75%,2.163463e+07,1.190500,2.687800,75.000000,6.766325e+09,7.000000,17.000000,4.000000
max,2.911215e+07,1.249900,2.749900,1004.000000,9.980711e+09,10.000000,23.000000,6.000000


In [22]:
"""
    groupby('place_id')：按照place_id这一列做分组，把相同place_id的所有行归到一组
    .count()：分组之后的聚合操作，统计每一组内，每一列非缺失值的数量
    count()不是统计计数，是统计每一列不为NaN的元素个数
    
    这里data的数据类型是 DataFrame
"""
place_count = data.groupby('place_id').count()
place_count

,row_id,x,y,accuracy,day,hour,weekday
place_id,,,,,,,
1012023972,1,1,1,1,1,1,1
1057182134,1,1,1,1,1,1,1
1059958036,3,3,3,3,3,3,3
1085266789,1,1,1,1,1,1,1
1097200869,1044,1044,1044,1044,1044,1044,1044
...,...,...,...,...,...,...,...
9904182060,1,1,1,1,1,1,1
9915093501,1,1,1,1,1,1,1
9946198589,1,1,1,1,1,1,1


In [23]:
place_count['x'].describe()

count     805.000000
mean       22.000000
std        88.955632
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max      1044.000000
Name: x, dtype: float64

In [24]:
"""
    .reset_index()：将索引place_id恢复为普通字段，重新生成默认0开始的整数行索引
    如果不写reset_index()，place_id仍然是索引，后续取tf['place_id']会报错
    place_count.row_id等价于place_count['row_id']，点写法要求列名不能有空格、不能是关键字
"""
tf = place_count[place_count.row_id > 3].reset_index()
tf

,place_id,row_id,x,y,accuracy,day,hour,weekday
0,1097200869,1044,1044,1044,1044,1044,1044,1044
1,1228935308,120,120,120,120,120,120,120
2,1267801529,58,58,58,58,58,58,58
3,1278040507,15,15,15,15,15,15,15
4,1285051622,21,21,21,21,21,21,21
...,...,...,...,...,...,...,...,...
234,9741307878,5,5,5,5,5,5,5
235,9753855529,21,21,21,21,21,21,21
236,9806043737,6,6,6,6,6,6,6
237,9809476069,23,23,23,23,23,23,23


In [25]:
data = data[data['place_id'].isin(tf.place_id)]
data.shape

(16918, 8)

In [26]:
y = data['place_id']
print(type(y))                                          # <class 'pandas.Series'>

x = data.drop(['place_id'], axis=1)
x = x.drop(['row_id'], axis=1)
print(x.shape)
print(x.columns)

<class 'pandas.Series'>
(16918, 6)
Index(['x', 'y', 'accuracy', 'day', 'hour', 'weekday'], dtype='str')


In [27]:
"""
train_test_split：sklearn数据集划分工具
x：全部特征集；y：全部标签
random_state=1：随机种子，固定随机打乱，每次运行划分结果完全一样
返回顺序：训练特征、测试特征（feature：特征值）、训练标签（target：目标值）、测试标签
"""
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=1)
"""
    std = StandardScaler()
        创建标准化转换器对象，此时只是创建对象，什么计算都没做
    std.fit(x_trian)
        在训练集上学习：
            计算训练集每一列特征的均值 std.mean_
            计算训练集每一列特征的方差 std.var_
    std.transform(x_train)
        套用公式做Z-score标准化：
            z = (x - μ) / σ
"""
std = StandardScaler()
x_train = std.fit_transform(x_train)
print(std.mean_)
print(std.var_)
print('--' * 50)

x_test = std.transform(x_test)
# 以下输出的均值与方差与上面一致，因为上面以及fit过了，已经计算过均值和方差了，所以此时输出的均值与方差与上述一致
print(std.mean_)
print(std.var_)

[ 1.12295735  2.63237278 81.34938525  5.10064628 11.44293821  3.10135561]
[5.98489138e-03 4.86857391e-03 1.19597480e+04 7.32837915e+00
 4.83742660e+01 2.81838404e+00]
----------------------------------------------------------------------------------------------------
[ 1.12295735  2.63237278 81.34938525  5.10064628 11.44293821  3.10135561]
[5.98489138e-03 4.86857391e-03 1.19597480e+04 7.32837915e+00
 4.83742660e+01 2.81838404e+00]


In [28]:
# 如果样本在特征空间中的k个最相似的样本中 大多数 数据某一类别，则该样本也属于这个类别
knn = KNeighborsClassifier(n_neighbors=6)
"""
KNN的fit做的事情：
    仅仅把训练集特征、标签保存到模型内部
KNN是惰性学习（lazy learning）:
    fit(): 不计算距离、不构建模型，只是把样本保存起来
    predict()预测阶段真正计算距离，找邻居
"""
knn.fit(x_train, y_train)
"""
理论逻辑：
    1. 对每个测试样本，计算该测试样本到训练集全部样本的距离
    2. 距离从小到大排序，取最近 k=6 个邻居
    3. 看这6个邻居的标签，投票，多数为预测类别
sklearn实际工程表现：
    KNeighborsClassifier默认用 algorithm='auto':
        样本少：直接暴力穷举距离（brute暴力搜索）
        样本量大：自动构建KD-Tree/Ball-Tree树索引结构，利用空间树剪枝，不需要跟所有训练样本计算距离，跳过大量不可能是最近邻的点，加速查找
"""
y_predict = knn.predict(x_test)

print('预测的目标签到位置为：', y_predict[0: 10])
print('真实测试集目标签到位置：', y_test[0: 10])
print(type(y_test))                                             # <class 'pandas.Series'>
print('--' * 50)
print('预测的准确率：', knn.score(x_test, y_test))
# print(y_predict)
# y_test

预测的目标签到位置为： [5689129232 1097200869 2355236719 9632980559 6424972551 4022692381
 8048985799 6683426742 1435128522 3312463746]
真实测试集目标签到位置： 16751286    1893548673
12423167    1097200869
7517023     6097504486
4400015     9632980559
26212472    6424972551
7089828     4022692381
10935607    2327054745
25025511    3533177779
27755137    1435128522
19678934    3312463746
Name: place_id, dtype: int64
<class 'pandas.Series'>
----------------------------------------------------------------------------------------------------
预测的准确率： 0.484160756501182


In [29]:
print(max(time_value))

1970-01-10 02:23:38


In [34]:
"""
n_neighbors=5：K-近邻算法的K，代表取多少个最近样本参与投票
    k太小：容易过拟合，对噪声敏感
    k太大：容易欠拟合，把很远样本拉进来干扰结果
weights权重模式：
    uniform：默认，全部邻居同等权重投票
    distance：距离越近权重越高，距离远的邻居话语权变小
全部组合数量：
    5个k值 x 2种权重 = 10套参数组合
    GridSearchCV会全部跑一遍，用交叉验证，挑验证集效果最好的那套参数
"""
param = {'n_neighbors': [3, 5, 10, 12, 15], 'weights': ['uniform', 'distance']}

# 进行网格搜索，cv=3是3折交叉验证，用其中2折训练，1折验证（把训练集等分3份，其中2份作训练，1份作验证，循环交叉作3次，即每1等分都作1次验证集）
gc = GridSearchCV(knn, param_grid=param, cv=3)
"""
将x_train分为训练集、验证集（没有测试集）
    全部搜索、交叉验证，只允许使用x_train，全程看不见x_test
    fit内部遍历param_grid里面所有超参数组合，每组做cv=3折验证
    保存每组的验证分数，选出平均验证得分最高的那一套超参数
    选好最优参数后，内部会重新在全部x_train上训练一个最终最优模型，存放在gc.best_estimator_
"""
gc.fit(x_train, y_train)

# 这里是选择好了最好的模型后的测试集上的准确率
print('在测试集上准确率：', gc.score(x_test, y_test))                    # 在测试集上准确率： 0.49763593380614657
print('在交叉验证中最好的结果：', gc.best_score_)                         # 在交叉验证中最好的结果： 0.4816362349278435
print('选择最好的模型是：', gc.best_estimator_)
# 选择最好的模型是： KNeighborsClassifier(n_neighbors=12, weights='distance')

# cv_results_是GridSearchCV.fit()完成之后生成的字典，记录每一组超参数对应的全部交叉验证结果
print('每个超参数每次交叉验证的结果：', gc.cv_results_)

/Users/qingjiabu/Library/Python/3.12/lib/python/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


在测试集上准确率： 0.49763593380614657
在交叉验证中最好的结果： 0.4816362349278435
选择最好的模型是： KNeighborsClassifier(n_neighbors=12, weights='distance')
每个超参数每次交叉验证的结果： {'mean_fit_time': array([0.00327094, 0.0031004 , 0.00305136, 0.00285101, 0.00304548,
       0.00304429, 0.0031573 , 0.003215  , 0.00326228, 0.00312495]), 'std_fit_time': array([1.37598759e-04, 5.65543075e-05, 1.40347000e-04, 9.03630529e-05,
       1.52120562e-04, 1.92674307e-04, 5.00044092e-05, 5.29129321e-05,
       1.48860664e-04, 5.27560932e-05]), 'mean_score_time': array([0.02246642, 0.03849761, 0.02629066, 0.04544536, 0.03744149,
       0.06710839, 0.04527696, 0.07297111, 0.05357202, 0.08316596]), 'std_score_time': array([0.00080058, 0.00320293, 0.00084141, 0.00206589, 0.00095797,
       0.00444898, 0.00097673, 0.00411861, 0.00231601, 0.0036457 ]), 'param_n_neighbors': masked_array(data=[3, 3, 5, 5, 10, 10, 12, 12, 15, 15],
             mask=[False, False, False, False, False, False, False, False,
                   False, False],
       

In [35]:
"""
朴素贝叶斯进行文本分类
"""
news = fetch_20newsgroups(subset='all', data_home='data')

print(len(news.data))
print(type(news.data))                                      # <class 'list'>
print('--' * 50)

print(news.data[0])
print('--' * 50)

print(news.target)
print(np.unique(news.target))                               # [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
print(news.target_names)

18846
<class 'list'>
----------------------------------------------------------------------------------------------------
From: Mamatha Devineni Ratnam <mr47+@andrew.cmu.edu>
Subject: Pens fans reactions
Organization: Post Office, Carnegie Mellon, Pittsburgh, PA
Lines: 12
NNTP-Posting-Host: po4.andrew.cmu.edu



I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a lot of
fun in the next couple of games since the Pens are going to beat the pulp out of Jersey anyway. I was very disappointed not to see the Islanders lose the final
regular season game.

/Users/qingjiabu/Library/Python/3.12/lib/python/site-packages/sklearn/datasets/_twenty_newsgroups.py:310: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  cache = pickle.loads(uncompressed_content)


In [36]:
x_train, x_test, y_train, y_test = train_test_split(news.data, news.target, test_size=0.25, random_state=1)

tf = TfidfVectorizer()
x_train = tf.fit_transform(x_train)
print(type(x_train))                                    # <class 'scipy.sparse._csr.csr_matrix'>
print(len(tf.get_feature_names_out()))

<class 'scipy.sparse._csr.csr_matrix'>
153196


In [99]:
print(tf.get_feature_names_out()[100000])

murky


In [100]:
print(tf.get_feature_names_out()[0: 10])

['00' '000' '0000' '00000' '0000000004' '0000000005' '0000000667'
 '0000001200' '000003' '000005102000']


In [101]:
print(tf.get_feature_names_out()[100000: 100002])

['murky' 'murmurs']


In [103]:
import time

# 进行朴素贝叶斯算法的预测，alpha是拉普拉斯平滑系数，分子和分母加上一个系数，分母加alpha*特征词数目
mlt = MultinomialNB(alpha=1.0)

print(x_train.toarray())
start = time.time()
mlt.fit(x_train, y_train)
end = time.time()
end-start

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


0.0592198371887207

In [104]:
x_test = tf.transform(x_test)
print(len(tf.get_feature_names_out()))

153196


In [106]:
start = time.time()
y_predict = mlt.predict(x_test)

print(f'预测的前面10篇文章类别为：{y_predict[0: 10]}')
print(f'准确率为：{mlt.score(x_test, y_test)}')
end = time.time()
end - start

预测的前面10篇文章类别为：[16 19 18  1  9 15  1  2 16 13]
准确率为：0.8518675721561969


0.059800148010253906

In [107]:
len(y_predict)

4712

In [109]:
"""
    y_test: 真实标签（ground_truth）
    y_predict: 模型预测出来的标签
    target_names=news.target_names：类别名称，把数字0/1/2、、、替换成真实类别字符串，比如新闻数据集的类别名字
        没有target_names时，输出只会显示数字类别，加上后显示文字类别
"""
print(f'每个类别的精确率和召回率：\n{classification_report(y_test, y_predict, target_names=news.target_names)}')

每个类别的精确率和召回率：
                          precision    recall  f1-score   support

             alt.atheism       0.91      0.77      0.83       199
           comp.graphics       0.83      0.79      0.81       242
 comp.os.ms-windows.misc       0.89      0.83      0.86       263
comp.sys.ibm.pc.hardware       0.80      0.83      0.81       262
   comp.sys.mac.hardware       0.90      0.88      0.89       234
          comp.windows.x       0.92      0.85      0.88       230
            misc.forsale       0.96      0.67      0.79       257
               rec.autos       0.90      0.87      0.88       265
         rec.motorcycles       0.90      0.95      0.92       251
      rec.sport.baseball       0.89      0.96      0.93       226
        rec.sport.hockey       0.95      0.98      0.96       262
               sci.crypt       0.76      0.97      0.85       257
         sci.electronics       0.84      0.80      0.82       229
                 sci.med       0.97      0.86      0.91      

In [110]:
max(y_test), min(y_test)

(np.int32(19), np.int32(0))

In [112]:
"""
    np.where(条件, 满足条件的值, 不满足条件的值)
    y_test1 = np.where(y_test == 5, 1, 0)
    逻辑：
    如果 y_test 的元素等于5 → 赋值为1
    其他所有类别 → 赋值为0
    作用：把多分类标签，转成二分类任务：类别5为正例(1)，其余全部负例(0)
"""
# 将是否为第五个类别分别作为 1 和 0
y_test1 = np.where(y_test == 5, 1, 0)
print(y_test1.sum())
y_predict1 = np.where(y_predict == 5, 1, 0)
print(y_predict1.sum())

print(f'AUC指标：{roc_auc_score(y_test1, y_predict1)}')

230
214
AUC指标：0.924078924393225


In [113]:
y_test1, y_predict1

(array([0, 0, 0, ..., 0, 0, 0], shape=(4712,)),
 array([0, 0, 0, ..., 0, 0, 0], shape=(4712,)))

In [115]:
# 测试值为0，而预测值为1
FP = np.where((np.array(y_test1) - np.array(y_predict1)) == -1, 1, 0).sum()
# 所有预测为Positive中减去伪正例即为真正例
TP = y_predict1.sum() - FP
print(TP)
# 测试值为1而预测值为0，伪反例
FN = np.where((np.array(y_test1) - np.array(y_predict1)) == 1, 1, 0).sum()
print(FN)
# 测试集中所有反例减去伪正例即为真反例的数量
TN = np.where(y_test1 == 0, 1, 0).sum() - FP
print(TN)

196
34
4464


In [118]:
# 精确率
TP / (TP + FP)

np.float64(0.9158878504672897)

In [119]:
# 召回率
TP / (TP + FN)

np.float64(0.8521739130434782)

In [117]:
# F1-score
2*TP / (2*TP + FN)

np.float64(0.92018779342723)

In [120]:
del news
del x_train
del x_test
del y_test
del y_predict
del tf

# 3 决策树

In [37]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

In [122]:
np.log2(1 / 32)

np.float64(-5.0)

In [123]:
1 / 2 * np.log2(1 / 2) + 1 / 2 * np.log2(1 / 2)

np.float64(-1.0)

In [124]:
1 / 3 * np.log2(1 / 3) + 2 / 3 * np.log2(2 / 3)

np.float64(-0.9182958340544896)

In [125]:
0.01 * np.log2(0.01) + 0.99 * np.log2(0.99)

np.float64(-0.08079313589591118)

In [38]:
"""
决策树对泰坦尼克号进行预测生死
"""
titan = pd.read_csv('./data/titanic.txt')
titan.info()

<class 'pandas.DataFrame'>
RangeIndex: 1313 entries, 0 to 1312
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   row.names  1313 non-null   int64  
 1   pclass     1313 non-null   str    
 2   survived   1313 non-null   int64  
 3   name       1313 non-null   str    
 4   age        633 non-null    float64
 5   embarked   821 non-null    str    
 6   home.dest  754 non-null    str    
 7   room       77 non-null     str    
 8   ticket     69 non-null     str    
 9   boat       347 non-null    str    
 10  sex        1313 non-null   str    
dtypes: float64(1), int64(2), str(8)
memory usage: 113.0 KB


In [39]:
"""
[['pclass', 'age', 'sex']]
    内层列表：'pclass', 'age', 'sex'：指定要取出的列名
    外层[]：pandas语法，选中多列
"""
x =  titan[['pclass', 'age', 'sex']]

y = titan['survived']
print(x.info())
print('--' * 50)
x.describe(include='all')                                   # pclass: passenger class

<class 'pandas.DataFrame'>
RangeIndex: 1313 entries, 0 to 1312
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pclass  1313 non-null   str    
 1   age     633 non-null    float64
 2   sex     1313 non-null   str    
dtypes: float64(1), str(2)
memory usage: 30.9 KB
None
----------------------------------------------------------------------------------------------------


,pclass,age,sex
count,1313,633.000000,1313
unique,3,NaN,2
top,3rd,NaN,male
freq,711,NaN,850
mean,NaN,31.194181,NaN
std,NaN,14.747525,NaN
min,NaN,0.166700,NaN
25%,NaN,21.000000,NaN
50%,NaN,30.000000,NaN
75%,NaN,41.000000,NaN


In [129]:
# 对缺失值进行处理，填为均值
mean=x['age'].mean()
"""
x.loc[:, 'age']
    ,前面：:代表所有行
    ,后面: 'age'代表age这一列
    逗号分隔行条件，列条件，loc语法：loc[行，列]
"""
x.loc[:, 'age']=x.loc[:, 'age'].fillna(mean)
x.info()                                                            # pclass: passenger class

<class 'pandas.DataFrame'>
RangeIndex: 1313 entries, 0 to 1312
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pclass  1313 non-null   str    
 1   age     1313 non-null   float64
 2   sex     1313 non-null   str    
dtypes: float64(1), str(2)
memory usage: 30.9 KB


In [130]:
# 分割数据集到训练集和测试集
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=4)
print(x_train.head())                                                   # pclass: passenger class

    pclass        age     sex
598    2nd  30.000000    male
246    1st  62.000000    male
905    3rd  31.194181  female
300    1st  31.194181  female
509    2nd  64.000000    male


In [131]:
type(x_train)

pandas.DataFrame

In [132]:
sum(y_train)

334

In [133]:
x_train[x_train['sex'] == 'female'].count()                             # pclass: passenger class 座舱等级

pclass    341
age       341
sex       341
dtype: int64

In [134]:
y_train

598     0
246     0
905     0
300     0
509     0
       ..
360     0
709     0
439     0
174     0
1146    0
Name: survived, Length: 984, dtype: int64

In [136]:
z = x_train.copy()
z['survived'] = y_train
"""
    z['sex'] == 'male' 布尔筛选：选出性别为男性的所有行
    再取出男性行里survived列（生存标签：1存活，0死亡）
    value_counts()统计该列各个取值有多少样本
"""
z[z['sex'] == 'male']['survived'].value_counts()

survived
0    539
1    104
Name: count, dtype: int64

In [137]:
x_train

,pclass,age,sex
598,2nd,30.000000,male
246,1st,62.000000,male
905,3rd,31.194181,female
300,1st,31.194181,female
509,2nd,64.000000,male
...,...,...,...
360,2nd,31.194181,male
709,3rd,28.000000,male
439,2nd,34.000000,male
174,1st,46.000000,male


In [138]:
"""
    orient='records'：把DataFrame每一行，转为一个字典
    每个字典的key=列名，value=该行对应的值
    返回列表，列表里面装一行一个dict；常用于DictVectorizer类别特征转换
"""
x_train.to_dict(orient='records')

[{'pclass': '2nd', 'age': 30.0, 'sex': 'male'},
 {'pclass': '1st', 'age': 62.0, 'sex': 'male'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'female'},
 {'pclass': '1st', 'age': 31.19418104265403, 'sex': 'female'},
 {'pclass': '2nd', 'age': 64.0, 'sex': 'male'},
 {'pclass': '1st', 'age': 31.19418104265403, 'sex': 'female'},
 {'pclass': '3rd', 'age': 24.0, 'sex': 'female'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '2nd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '3rd', 'age': 21.0, 'sex': 'male'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '2nd', 'age': 23.0, 'sex': 'female'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'male'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'female'},
 {'pclass': '3rd', 'age': 31.19418104265403, 'sex': 'female'},
 {'pclass': '1st', 'age': 4

In [139]:
# 进行处理（特征工程）特征 -> 类别 -> one-hot编码
dict = DictVectorizer(sparse=False)

x_train = dict.fit_transform(x_train.to_dict(orient='records'))
print(type(x_train))
print(dict.get_feature_names_out())
print('--' * 50)
x_test = dict.transform(x_test.to_dict(orient='records'))
print(x_train)

<class 'numpy.ndarray'>
['age' 'pclass=1st' 'pclass=2nd' 'pclass=3rd' 'sex=female' 'sex=male']
----------------------------------------------------------------------------------------------------
[[30.          0.          1.          0.          0.          1.        ]
 [62.          1.          0.          0.          0.          1.        ]
 [31.19418104  0.          0.          1.          1.          0.        ]
 ...
 [34.          0.          1.          0.          0.          1.        ]
 [46.          1.          0.          0.          0.          1.        ]
 [31.19418104  0.          0.          1.          0.          1.        ]]


In [42]:
dec = DecisionTreeClassifier()

dec.fit(x_train, y_train)

print(f'预测的准确率：{dec.score(x_test, y_test)}')
# 
export_graphviz(dec, out_file='tree.dot', feature_names=['age', 'pclass=1st', 'pclass=2nd', 'pclass=3rd', 'female', 'male'])

ValueError: could not convert string to float: 'From: Seth Adam Eliot <se08+@andrew.cmu.edu>\nSubject: Re: 2ND AMENDMENT DEAD - GOOD !\nOrganization: Doctoral student, Materials Science and Engineering, Carnegie Mellon, Pittsburgh, PA\nLines: 58\nNNTP-Posting-Host: po3.andrew.cmu.edu\nIn-Reply-To: <1993Apr18.001319.2340@gnv.ifas.ufl.edu>\n\nExcerpts from netnews.talk.politics.guns: 18-Apr-93 2ND AMENDMENT DEAD -\nGOOD ! by jrm@gnv.ifas.ufl.edu \n> Yea, there are millions of cases where yoy *say* that firearms\n> \'deter\' criminals. Alas, this is not provable. I think that that\n> there are actually *few* cases where this is so. \n\nexcerpted from a letter I wrote a while ago:\n\n     Although less apparent to those who have not researched\nthe facts, personal protection is as legitimate a reason  as\nsport for the private citizen to own a gun.  The most recent\nresearch  is  that  of Dr. Gary Kleck of the  Florida  State\nUniversity  School of Criminology.1  He found that  handguns\na

In [141]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=4)
dict = DictVectorizer(sparse=False)

x_train = dict.fit_transform(x_train.to_dict(orient='records'))
x_test = dict.transform(x_test.to_dict(orient='records'))

"""

"""
dec = DecisionTreeClassifier(max_depth=7, min_impurity_decrease=0.01)

dec.fit(x_train, y_train)

print(f'预测准确率：{dec.score(x_test, y_test)}')

export_graphviz(dec, out_file='tree1.dot', 
                feature_names=dict.get_feature_names_out())

预测准确率：0.8206686930091185


In [142]:
y_train.shape

(984,)

In [143]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=4)
dict = DictVectorizer(sparse=False)

x_train = dict.fit_transform(x_train.to_dict(orient='records'))
x_test = dict.transform(x_test.to_dict(orient='records'))

In [144]:
rf = RandomForestClassifier(n_jobs=-1)

param = {'n_estimators': [1500, 2000, 5000], 'max_depth': [2, 3, 5, 8, 15, 25]}

gc = GridSearchCV(rf, param_grid=param, cv=3)

gc.fit(x_train, y_train)

print(f'准确率：{gc.score(x_test, y_test)}')
print(f'查看选择的参数模型：{gc.best_params_}')
print(f'选择最好的模型是：{gc.best_estimator_}')
print(f'每个超参数每次交叉验证的结果：{gc.cv_results_}')

准确率：0.8328267477203647
查看选择的参数模型：{'max_depth': 3, 'n_estimators': 1500}
选择最好的模型是：RandomForestClassifier(max_depth=3, n_estimators=1500, n_jobs=-1)
每个超参数每次交叉验证的结果：{'mean_fit_time': array([1.15862211, 1.36947727, 3.41393797, 1.06944871, 1.38501517,
       3.38806677, 1.03037421, 1.37866092, 3.41014036, 1.03435175,
       1.41873487, 3.58981411, 1.07797631, 1.3974208 , 3.51661777,
       1.03120224, 1.37842751, 3.43157236]), 'std_fit_time': array([0.18491276, 0.027932  , 0.03483594, 0.06430737, 0.00477617,
       0.02716108, 0.01134655, 0.01682714, 0.02309131, 0.0027313 ,
       0.02830755, 0.06013989, 0.00844693, 0.02556143, 0.03987794,
       0.02298035, 0.02084159, 0.09775451]), 'mean_score_time': array([0.10400065, 0.12506763, 0.31063143, 0.104273  , 0.1459705 ,
       0.32571395, 0.11167304, 0.14152137, 0.35316102, 0.11381189,
       0.15124035, 0.40615161, 0.12896538, 0.16878732, 0.3727843 ,
       0.1177272 , 0.15505568, 0.37182458]), 'std_score_time': array([0.00669949, 0.00178195